# Document-Level Relation Extraction with Localized Context Pooling (LCoP)
## Lifestyle Factor–Disease Relation Extraction on the LSD600 Benchmark

### Overview & Motivation:
- **Resolving the 180-Token Bottleneck:** The base paper (*Database (Oxford)* 2025) used a 180-token window that caused **40% of all False Negatives** across cross-sentence pairs.
- **Document-Level Modeling (512 tokens):** Covers >91% of full abstracts and 92.5% of all candidate pairs while retaining identical entity masking (`[unused1]` focus pair, `[unused2]` context entities) for strict ablation isolation.
- **Localized Context Pooling (LCoP):** Synthesizes cross-entity attention evidence bridges $A_{e1, e2} = A_{e1} \odot A_{e2}$ and pair-specific bilinear queries to pool connecting context tokens into $c_{e1, e2}$.
- **Direct Official Evaluation:** Evaluates every checkpoint against official `evalsorel.py` to compare directly with the reproduced baseline (**F1 = 68.5%**, P = 77.3%, R = 61.6%).
- **Colab Safeguards:** Automatic checkpoint resuming (`--resume`), intra-epoch saving every 500 steps (`--save-steps 500`), 2x faster throughput (`batch-size 8`), storage-safe saving (<2.8 GB total), early stopping (`--patience 4`), and persistent metric export (`test_results.json`, `training_history.json`).

In [ ]:
# Step 1: Check GPU allocation and available VRAM (T4, V100, A100, or L4)
!nvidia-smi

In [ ]:
# Step 2: Mount Google Drive & Navigate to Workspace
from google.colab import drive
drive.mount('/content/drive')

import os

candidate_paths = [
    '/content/drive/MyDrive/research/Students/Project',
    '/content/drive/My Drive/research/Students/Project',
    '/content/drive/MyDrive/research/Students/AtashBar/Project',
    '/content/drive/My Drive/research/Students/AtashBar/Project',
    '/content/drive/MyDrive/research/Students',
    '/content/drive/My Drive/research/Students',
    '/content/drive/MyDrive/Students/AtashBar/Project',
    '/content/drive/My Drive/Students/AtashBar/Project',
    '/content/drive/MyDrive/Project',
    '/content/drive/My Drive/Project'
]

project_dir = None
for p in candidate_paths:
    if os.path.exists(os.path.join(p, 'src/training/train_doc_level_lcop.py')):
        project_dir = p
        break

# Fallback search if located in subfolder
if not project_dir:
    for root_dir in ['/content/drive/MyDrive/research', '/content/drive/MyDrive']:
        if os.path.exists(root_dir):
            for root, dirs, files in os.walk(root_dir):
                if 'train_doc_level_lcop.py' in files and 'src/training' in root.replace('\\', '/'):
                    project_dir = os.path.abspath(os.path.join(root, '..', '..'))
                    break
        if project_dir:
            break

if project_dir:
    os.chdir(project_dir)
    print(f"✅ Successfully located project directory: {os.getcwd()}")
else:
    raise FileNotFoundError("Project folder not found. Please verify Google Drive path.")

In [ ]:
# Step 3: Install Required Dependencies & Configure Environment
!pip install -q --upgrade transformers tokenizers scikit-learn
import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
print("✅ Environment configured successfully.")

In [ ]:
# Step 4: Verify Environment and GPU Acceleration
import torch
from transformers import AutoTokenizer
from src.models.localized_context_pooling import DocumentLevelLCoPExtractor, LCoPConfig
from src.training.train_doc_level_lcop import DocLevelLCoPDataset, collate_lcop_batch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    print(f"Allocated VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
# Step 5: Check Candidate Pair Statistics across Splits (512-token context)
tokenizer = AutoTokenizer.from_pretrained("models/pretrained_roberta_lsd600", use_fast=True)
train_ds = DocLevelLCoPDataset("data/raw/LSD600/train", tokenizer, max_seq_length=512, is_training=True)
dev_ds = DocLevelLCoPDataset("data/raw/LSD600/dev", tokenizer, max_seq_length=512, is_training=False)
test_ds = DocLevelLCoPDataset("data/raw/LSD600/test", tokenizer, max_seq_length=512, is_training=False)

print(f"Candidate Pairs (512 tokens): Train={len(train_ds)}, Dev={len(dev_ds)}, Test={len(test_ds)}")

In [ ]:
# Step 6: Dedicated Standalone Test Set Evaluation with Official evalsorel.py
# Evaluate current best checkpoint on the held-out test set (~1-2 mins)
!python -m src.training.train_doc_level_lcop \
  --test-dir data/raw/LSD600/test \
  --model-path models/pretrained_roberta_lsd600 \
  --checkpoint-path models/checkpoints_doc_level_lcop/best_lcop_model.pt \
  --output-dir models/checkpoints_doc_level_lcop \
  --max-seq-length 512 \
  --eval-batch-size 32 \
  --eval-only

In [ ]:
# Step 7: Train / Resume Epoch 4 & 5 (Seamless Mid-Epoch Resumption)
# - AUTO-RESUME: Automatically detects latest_checkpoint.pt and resumes at the exact step (Step 3500 of Epoch 4)
# - ZERO LOST PROGRESS: Only finishes the remaining ~1,000 steps of Epoch 4 (~8-9 mins), then moves to Epoch 5
# - INTRA-EPOCH SAVING: Saves every 500 steps (~7-10 mins) so any future disconnects are fully protected
# - Automatically evaluates Dev set after each epoch with official evalsorel.py
# - Updates best_lcop_model.pt whenever Dev F1 beats previous record
# - Runs final test set evaluation at completion
!python -m src.training.train_doc_level_lcop \n  --train-dir data/raw/LSD600/train \n  --dev-dir data/raw/LSD600/dev \n  --test-dir data/raw/LSD600/test \n  --model-path models/pretrained_roberta_lsd600 \n  --output-dir models/checkpoints_doc_level_lcop \n  --max-seq-length 512 \n  --pooling-mode hybrid \n  --epochs 5 \n  --batch-size 8 \n  --grad-accum-steps 2 \n  --eval-batch-size 32 \n  --lr 2e-5 \n  --threshold 0.5 \n  --save-steps 500 \n  --save-interval-minutes 15 \n  --num-workers 2 \n  --fp16 \n  --resume


In [ ]:
# Step 8: View & Compare Evaluation Results against the Base Paper
import json
import pandas as pd

res_path = "models/checkpoints_doc_level_lcop/test_results.json"
if os.path.exists(res_path):
    with open(res_path, "r") as f:
        results = json.load(f)

    print("=" * 75)
    print("LSD600 BENCHMARK: BASELINE vs. DOCUMENT-LEVEL LCoP (OURS)")
    print("=" * 75)

    summary_df = pd.DataFrame([
        {
            "Model": "Base Paper (RoBERTa 180 tokens)",
            "Precision (%)": 77.3,
            "Recall (%)": 61.6,
            "F1-Score (%)": 68.5,
            "Delta F1 (%)": "Baseline",
        },
        {
            "Model": "Document-Level LCoP (512 tokens, Ours)",
            "Precision (%)": round(results["test_precision"], 1),
            "Recall (%)": round(results["test_recall"], 1),
            "F1-Score (%)": round(results["test_f1"], 1),
            "Delta F1 (%)": f"{round(results['test_f1'] - 68.5, 1):+0.1f}%",
        },
    ])
    display(summary_df)

    if "per_class" in results and results["per_class"]:
        print("\n" + "-" * 75)
        print("PER-CLASS PERFORMANCE BREAKDOWN:")
        print("-" * 75)
        per_class_df = pd.DataFrame.from_dict(results["per_class"], orient="index")
        display(per_class_df)
else:
    print(f"Results file not found at {res_path}. Please complete training/evaluation first.")